# Track B / Phase B1 — SVR–GARCH CUSUM 기본설정

이 노트북은 Lee, Kim & Lee (2020)의 **영어 원문을 실행 계약으로 고정**하는 B00 문서다. Eq. (1)–(9), §3 CUSUM, Tables 1–6, Figures 1–4의 근거와 gap을 검증하지만 모델 적합, Monte Carlo, 외부 데이터 다운로드는 수행하지 않는다.

- Source of truth: 결합 PDF의 물리 30–45쪽(영어 원문 16쪽)
- Phase gate: 사용자 승인 전 B2 금지
- 범위 제외: ETF 적용, 온라인 탐지, 신규 알고리즘

In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib.metadata as metadata
import platform
import sys

import numpy as np
import yaml

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'config' / 'paper2020_contract.yaml').exists():
            return candidate
    raise FileNotFoundError('config/paper2020_contract.yaml을 찾지 못했습니다.')

REPO_ROOT = find_repo_root(Path.cwd())
CONTRACT_PATH = REPO_ROOT / 'config' / 'paper2020_contract.yaml'
TRACE_PATH = REPO_ROOT / 'inherited_b1' / 'paper_traceability.csv'
contract = yaml.safe_load(CONTRACT_PATH.read_text(encoding='utf-8'))
print('repo_root:', REPO_ROOT)
print('phase:', contract['phase'])
print('status:', contract['status'])

## 1. 원문 식별과 무결성

한국어 연구판은 해설용이며 충돌 시 영어 원문이 우선한다. 아래 셀은 로컬 PDF가 감사한 snapshot과 같은지 확인한다.

In [ ]:
pdf_path = Path(contract['source']['local_pdf'])
assert pdf_path.exists(), f'원문 PDF 없음: {pdf_path}'
actual_sha = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
assert actual_sha == contract['source']['sha256']
assert contract['source']['english_original_physical_pages'] == [30, 45]
print('PDF SHA-256:', actual_sha)
print('English original: physical PDF pages 30–45')

## 2. Eq. (1)–(9) 추적 계약

- Eq. (1): $y_t=\sigma_t e_t$, $\sigma_t^2=g(y_{t-1}^2,\ldots,y_{t-p}^2,\sigma_{t-1}^2,\ldots,\sigma_{t-q}^2)$.
- Eq. (2)–(7): $\epsilon$-insensitive SVR의 loss, primal, Lagrangian, KKT, dual.
- Eq. (8): support-vector kernel 표현.
- Eq. (9): 과거 제곱 관측치와 과거 조건부분산으로 만든 feature vector.

원문 RBF는 $\exp[-\|x-y\|^2/(2\gamma^2)]$이므로 sklearn의 계수는 `1/(2*paper_gamma_squared)`다. 논문의 숫자를 sklearn `gamma`에 그대로 넣지 않는다.

In [ ]:
assert list(contract['equations']) == [f'eq_{i}' for i in range(1, 10)]
x = np.array([1.0, -2.0])
y = np.array([0.5, 1.0])
d2 = float(np.sum((x - y) ** 2))
for paper_gamma_squared in (0.1, 0.5, 1.0):
    gamma_sklearn = 1.0 / (2.0 * paper_gamma_squared)
    paper_kernel = np.exp(-d2 / (2.0 * paper_gamma_squared))
    sklearn_form = np.exp(-gamma_sklearn * d2)
    assert np.isclose(paper_kernel, sklearn_form, rtol=0, atol=1e-15)
print('Eq. inventory: 1–9 complete')
print('RBF coefficient mapping: numerically verified at 3 paper gamma² values')

## 3. 원문 5단계 SVR–GARCH 절차

1. 시간순 training/validation 분할과 grid 평가점 지정
2. $m=5$ 이동평균 proxy 구성. 주의: 시점 $t$의 proxy는 $y_t^2$를 포함한다.
3. proxy를 조건부분산 대신 사용하여 SVR 적합 및 $\hat\sigma_t^2$ 예측
4. validation에서 같은 proxy 공식을 사용해 MAE 계산
5. 최소 MAE 조합을 선택하고 training+validation에 재적합

Remark 2에 따라 반복적 variance 재적합은 exact protocol에서 제외한다. 정확한 grid, 경계 처리, test-time recursion, non-positive variance 처리는 원문 미명시 gap이다.

## 4. §3 retrospective CUSUM of squares 계약

표준화 잔차 $\hat e_t=y_t/\hat\sigma_t$의 제곱에 대해
$D_k=\sum_{t=1}^k\hat e_t^2-(k/n)\sum_{t=1}^n\hat e_t^2$를 만들고,
$\hat\tau_n^2=n^{-1}\sum\hat e_t^4-(n^{-1}\sum\hat e_t^2)^2$로 정규화한다.
$\hat T_n=\max_k|D_k|/(\sqrt n\hat\tau_n)$이며 5%에서 `T >= 1.3397`이면 기각한다. 위치는 별도 출력인 $\arg\max_k|D_k|$다.

In [ ]:
# B2 production 구현이 아니라 계약 fixture의 독립 산술 검산이다.
e = np.array(contract['cusum']['golden_test']['residuals'], dtype=float)
e2 = e**2
n = e.size
path = np.cumsum(e2) - (np.arange(1, n + 1) / n) * e2.sum()
tau2 = np.mean(e**4) - np.mean(e2) ** 2
stat = np.max(np.abs(path)) / (np.sqrt(n) * np.sqrt(tau2))
k_one_based = int(np.argmax(np.abs(path)) + 1)
gold = contract['cusum']['golden_test']
assert np.allclose(path, gold['centered_path'])
assert np.isclose(tau2, gold['tau_squared'])
assert np.isclose(stat, gold['statistic'])
assert k_one_based == gold['argmax_one_based']
reject = bool(stat >= contract['cusum']['critical_value'])
assert reject is gold['reject_at_5pct']
print({'D_k': path.tolist(), 'tau_squared': tau2, 'T': stat, 'argmax_1based': k_one_based, 'reject': reject})

## 5. §4 Tables 1–6 계약

- $n\in\{500,1000,2000\}$, 총 길이 $2n$
- 처음 $0.7n$ training, 다음 $0.3n$ validation, 나머지 $n$ testing
- 1,000회, 유의수준 0.05, iid standard normal innovation, $p=q=1$, $m=5$
- DGP: GARCH, AGARCH, GJR-GARCH, TGARCH(표준편차 recursion), log-linear GARCH
- Table 5 및 Table 6 Cases 1–2의 `N=20` 식은 원문 그대로면 양·음 극단 모두 `+20`이다. 부호보존 clipping은 별도 sensitivity다.

가장 큰 차단점은 H1의 'middle of the time series'가 정확한 index로 정의되지 않았다는 점이다. 생성된 $2n$의 중간은 testing 시작 경계이므로 testing 내부 변화점과 충돌한다.

## 6. §5 Figures 1–4 계약

| Series | Full period | Target change | Target T |
|---|---|---:|---:|
| S&P 500 | 2012-01-03–2016-09-30 | 2015-08-18 | 2.4751 |
| KOSPI | 2012-10-02–2019-06-28 | 2018-01-22 | 1.5081 |
| KRW/USD | 2012-01-02–2016-09-30 | 2014-09-26 | 4.2754 |

출처는 Investing.com이며 원 snapshot은 로컬에 없다. Figure 1은 앞쪽 절반 training 로그수익률의 ACF/PACF를 lag 25까지 표시한다. Figures 2–4는 각각 (a) 원 가격/지수와 (b) 로그수익률의 2개 패널이며, training은 노란 음영, 후반은 연두 음영, 검출점은 빨간 수직선이다. S&P 500의 전체 시작일과 예시 training 시작일(2012-04-02)은 원문 내부에서 충돌한다. 목표값을 맞추기 위한 사후 tuning은 금지한다.

In [ ]:
with TRACE_PATH.open(encoding='utf-8', newline='') as handle:
    trace = list(csv.DictReader(handle))
ids = {row['requirement_id'] for row in trace}
required = {*(f'EQ-{i:02d}' for i in range(1, 10)), *(f'TABLE-{i:02d}' for i in range(1, 7)), *(f'FIGURE-{i:02d}' for i in range(1, 5)), 'CUSUM-STAT', 'CUSUM-LOC'}
assert required <= ids
assert all(row['B1_status'] == 'SPECIFIED' for row in trace)
print(f'traceability rows: {len(trace)}')
print('required Eq/Table/Figure/CUSUM rows: complete')

## 7. 환경과 Phase gate

원 저자 환경은 Windows 10 / R 3.5.1 / `e1071` / `fGarch`다. 현재 환경은 WSL2 / Python 3.12.3 / scikit-learn 1.9.0이며 R과 `arch`가 없다. Python method-equivalent 결과와 R environment-parity 결과는 분리해야 한다.

In [ ]:
packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels', 'PyYAML', 'ipykernel', 'nbconvert', 'nbformat']
snapshot = {name: metadata.version(name) for name in packages}
print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('interpreter:', sys.executable)
print('packages:', snapshot)
print('B1_NOTE: no model fit, simulation, download, or file write was executed')

## 8. Phase B2 구현 계획 — 승인 대기

승인 후 순서는 (1) proxy/lag/kernel fixture, (2) SVR 선택과 경계 정책, (3) CUSUM golden tests, (4) 다섯 DGP smoke tests, (5) preregistered full protocol이다. 구현 전 `reproducibility_gap_register.md`의 B2 gate 항목을 사용자가 결정해야 한다.

**TRACK B / PHASE B1 ONLY — B2 NOT STARTED**

## 2차 감사 메모 (2026-10-11)
- 테스트 구간 분산 lag 정책 (A 관측 proxy / B 예측 분산) 은 원문에 명시되지 않음 → **UNRESOLVED**. 상세: `notebooks/01` §8, `reports/TEST_TIME_RECURSION_AUDIT.md`
- Table 6 TGARCH→AGARCH ω=1 의 post β 상속 결함 수정 (0.4→0.3, `reports/METHOD_FIDELITY_AUDIT.md` §4)
- 이 노트북은 B1 계약에 따라 출력 없이 저장된다.